# Zepto DRHP — Unit Economics & Supply Chain Analysis
**Source:** Zepto UDRHP-1 (SEBI, June 8 2026), Zomato Q4 FY26 investor presentation, Swiggy Q4 FY26 results

This notebook answers three supply chain questions:
1. Is Zepto's dark store network becoming more efficient over time?
2. How does Zepto's unit economics compare to Blinkit and Swiggy Instamart?
3. What is the path to store-level breakeven?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 1. Dark store productivity — the densification story

In [ ]:
opd = pd.DataFrame({
    'period': ['FY24', 'FY25', 'FY26', 'Q4 FY26'],
    'orders_per_day_per_store': [1325, 1565, 1677, 2140]
})

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(opd['period'], opd['orders_per_day_per_store'],
              color=['#B5D4F4','#378ADD','#185FA5','#0C447C'],
              width=0.5, zorder=3)
ax.set_ylabel('Orders / day / store')
ax.set_title('Dark store productivity: orders per day per store', fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_ylim(0, 2600)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='500')
ax.annotate('50% jump in\na single year', xy=(3, 2140), xytext=(2.3, 2300),
            arrowprops=dict(arrowstyle='->', color='#E24B4A'), color='#E24B4A', fontsize=9)
plt.tight_layout()
plt.savefig('../data/chart_store_productivity.png', bbox_inches='tight')
plt.show()
print('Insight: Fixed costs stay flat while orders scale — each incremental order is nearly pure margin.')

## 2. Competitor comparison — Zepto vs Blinkit vs Instamart

In [ ]:
# Sources: Zepto DRHP, Zomato Q4 FY26 investor deck, Swiggy Q4 FY26 results
competitors = pd.DataFrame({
    'company':       ['Blinkit',  'Zepto',   'Instamart'],
    'dark_stores':   [1544,       1139,      1062],
    'market_share':  [50,         29,        24],
    'aov_rs':        [709,        650,       619],   # Average Order Value
    'ebitda_status': ['Positive', 'Negative','Negative']
})

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['#185FA5', '#1D9E75', '#BA7517']

# Dark stores
axes[0].bar(competitors['company'], competitors['dark_stores'], color=colors, width=0.5)
axes[0].set_title('Dark store count', fontweight='bold')
axes[0].set_ylabel('Stores')
axes[0].yaxis.grid(True, linestyle='--', alpha=0.4)
for i, v in enumerate(competitors['dark_stores']):
    axes[0].text(i, v + 20, str(v), ha='center', fontsize=10)

# Market share
axes[1].bar(competitors['company'], competitors['market_share'], color=colors, width=0.5)
axes[1].set_title('Market share (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].yaxis.grid(True, linestyle='--', alpha=0.4)
for i, v in enumerate(competitors['market_share']):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)

# AOV
axes[2].bar(competitors['company'], competitors['aov_rs'], color=colors, width=0.5)
axes[2].set_title('Avg order value (₹)', fontweight='bold')
axes[2].set_ylabel('₹')
axes[2].yaxis.grid(True, linestyle='--', alpha=0.4)
for i, v in enumerate(competitors['aov_rs']):
    axes[2].text(i, v + 5, f'₹{v}', ha='center', fontsize=10)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Q-commerce competitive landscape: Blinkit vs Zepto vs Instamart (FY26)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/chart_competitor_comparison.png', bbox_inches='tight')
plt.show()

## 3. The ad revenue flywheel — Zepto's hidden margin engine

In [ ]:
ad = pd.DataFrame({
    'fiscal_year': ['FY24', 'FY25', 'FY26'],
    'ad_revenue_cr': [49, 651, 1636],
    'ad_pct_nrv': [1.1, None, 7.8]
})

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

bars = ax1.bar(ad['fiscal_year'], ad['ad_revenue_cr'],
               color=['#9FE1CB','#1D9E75','#0F6E56'], width=0.4, zorder=3)
ax1.set_ylabel('Ad revenue (₹ Crore)', color='#0F6E56')
ax1.set_ylim(0, 2200)
ax1.yaxis.grid(True, linestyle='--', alpha=0.3, zorder=0)

pct_vals = [1.1, None, 7.8]
pct_x = [0, 2]
pct_y = [1.1, 7.8]
ax2.plot(pct_x, pct_y, 'o--', color='#BA7517', linewidth=2, markersize=7, zorder=4)
ax2.set_ylabel('Ad rev as % of NRV', color='#BA7517')
ax2.set_ylim(0, 12)

for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f'₹{int(bar.get_height())}Cr', ha='center', fontsize=9)

ax1.set_title('Ad revenue: from side income to core business (33× growth in 2 years)',
              fontweight='bold', pad=12)
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig('../data/chart_ad_revenue.png', bbox_inches='tight')
plt.show()
print('Insight: Ad revenue is high-margin (no delivery cost). At 7.8% of NRV it is already subsidising last-mile operations.')

## 4. Path to breakeven — sensitivity analysis

In [ ]:
# At current trajectory, when does loss % of revenue hit zero?
loss_pct = pd.DataFrame({
    'fiscal_year': ['FY23', 'FY24', 'FY25', 'FY26'],
    'loss_pct': [63, 28, 42, 26]
})

# Simple linear extrapolation on FY24 and FY26 trend
years_numeric = [2024, 2026]
loss_values = [28, 26]
coeffs = np.polyfit(years_numeric, loss_values, 1)
breakeven_year = -coeffs[1] / coeffs[0]

future_years = list(range(2024, 2032))
projected = [max(0, coeffs[0]*y + coeffs[1]) for y in future_years]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(['FY23','FY24','FY25','FY26'], loss_pct['loss_pct'],
        'o-', color='#E24B4A', linewidth=2.5, markersize=7, label='Actual', zorder=4)
proj_labels = [f'FY{str(y)[2:]}' for y in future_years]
ax.plot(proj_labels, projected, 'o--', color='#888780', linewidth=1.5,
        markersize=5, label='Linear projection', zorder=3)
ax.axhline(0, color='#1D9E75', linewidth=1.5, linestyle=':', label='Breakeven')
ax.fill_between(proj_labels, projected, 0, alpha=0.07, color='#1D9E75')
ax.set_ylabel('Net loss as % of revenue')
ax.set_title('Path to breakeven: loss % of revenue trend & projection', fontweight='bold', pad=12)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../data/chart_breakeven_path.png', bbox_inches='tight')
plt.show()
print(f'Linear projection suggests revenue breakeven around FY{str(int(breakeven_year))[2:]}.')
print('Note: FY25 loss spike was due to aggressive dark store expansion — a one-time capex cycle, not structural deterioration.')